# Day 39 — 결론이 얼마나 튼튼한가: 민감도 분석

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-6/day-39.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-6/day-39.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — outlier 포함/제외에 따른 효과크기 변화

가상의 유전자 발현량(군당 6마리)입니다. 비행군의 마지막 값 13.9가 outlier 후보입니다.

In [ ]:
import numpy as np

def cohens_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    pooled_sd = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1))
                        / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / pooled_sd

# 가상 데이터: 어떤 유전자의 정규화 발현량 (군당 6마리)
ground = [10.1, 9.8, 10.3, 10.0, 9.9, 10.2]
flight = [10.4, 10.6, 10.2, 10.5, 10.3, 13.9]   # 마지막 값이 outlier 후보

print("=== 분석 A: 전체 포함 ===")
print("비행군 평균:", round(np.mean(flight), 2), "/ 중앙값:", round(np.median(flight), 2))
print("Cohen's d  :", round(cohens_d(flight, ground), 2))

print("=== 분석 B: outlier 후보(13.9) 제외 ===")
flight_wo = flight[:-1]
print("비행군 평균:", round(np.mean(flight_wo), 2), "/ 중앙값:", round(np.median(flight_wo), 2))
print("Cohen's d  :", round(cohens_d(flight_wo, ground), 2))

print("=== 평균 차이 vs 중앙값 차이 ===")
print("평균 차이   (포함/제외):", round(np.mean(flight) - np.mean(ground), 2), "/",
      round(np.mean(flight_wo) - np.mean(ground), 2))
print("중앙값 차이 (포함/제외):", round(np.median(flight) - np.median(ground), 2), "/",
      round(np.median(flight_wo) - np.median(ground), 2))

### 예시 2 — feature 정의를 바꿔 보기

걸음 변동성을 표준편차(SD), 변동계수(CV), 중앙값 절대편차(MAD) 세 가지로 정의해 PD군과 대조군의 효과크기를 비교합니다. 데이터는 코드 안에서 만든 가상의 값입니다.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

def cohens_d(a, b):
    pooled_sd = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1))
                        / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / pooled_sd

# 가상 데이터: 참가자 1명당 걸음 간격(stride interval, 초) 50개
records = []
for group, n, mean_sd in [("control", 12, 0.020), ("PD", 12, 0.028)]:
    for i in range(n):
        base = rng.normal(1.05, 0.08)                 # 사람마다 다른 평균 걸음 간격
        sd = abs(rng.normal(mean_sd, 0.008))          # 사람마다 다른 걸음 변동성
        strides = rng.normal(base, sd, size=50)
        records.append({
            "participant": f"{group}_{i:02d}",
            "group": group,
            "sd": strides.std(ddof=1),                           # 정의 1: 표준편차(초)
            "cv": strides.std(ddof=1) / strides.mean() * 100,    # 정의 2: 변동계수(%)
            "mad": np.median(np.abs(strides - np.median(strides))),  # 정의 3: 중앙값 절대편차(초)
        })
df = pd.DataFrame(records)

print("참가자 수:", df["group"].value_counts().to_dict())
for feature in ["sd", "cv", "mad"]:
    pd_vals = df.loc[df["group"] == "PD", feature].to_numpy()
    ct_vals = df.loc[df["group"] == "control", feature].to_numpy()
    print(f"{feature:>3}: Cohen's d = {cohens_d(pd_vals, ct_vals):.2f}")